# Soup AI Engineer take-home: run

Kaggle: Accelerator **GPU T4 x2** (only one GPU is used: `CUDA_VISIBLE_DEVICES=0`), **Internet on**.
Colab: Runtime → T4 GPU. Set `REPO_URL`, `COMMIT` and `BASE_WEIGHTS` below, then **Run All**.
Every cell that fails stops the notebook (assert).

In [ ]:
REPO_URL = "https://github.com/<you>/<repo>.git"  # public repository with this project at its root
COMMIT = ""            # required: the exact commit to run
BASE_WEIGHTS = "fp16"  # what merge.py merges into: "fp16" (original weights) or "nf4-dequant"
assert COMMIT, "set COMMIT to the commit you want to run"
assert BASE_WEIGHTS in ("fp16", "nf4-dequant")

## Setup

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4, even on Kaggle's T4 x2
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
WORK = ROOT / f"run_{COMMIT[:8]}_{time.strftime('%Y%m%dT%H%M%S')}"  # a fresh folder per Run All

def sh(cmd, cwd=None):
    """Run a shell command, stream its output into the cell, return the exit code."""
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.split("\n")[0]
print("GPU 0:", gpus)
assert "T4" in gpus, "this run must be on a T4"

assert sh(f"git clone -q {REPO_URL} {WORK}") == 0, "git clone failed"
assert sh(f"git checkout -q {COMMIT}", cwd=WORK) == 0, "git checkout failed"
sh("git log -1 --format='%H %s'", cwd=WORK)

assert sh(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=WORK) == 0, "pip install failed"
sh(f"{sys.executable} -m pip list 2>/dev/null | grep -iE '^(soup-cli|torch|transformers|peft|trl|bitsandbytes|accelerate|datasets) '")

## Tests

In [ ]:
assert sh(f"{sys.executable} -m pytest tests -q", cwd=WORK) == 0, "tests failed: stopping here"

## Prepare data, train, merge, quantize, generate (task 1)

`harness/smoke.py` runs `run_train.py` (prepare data + `soup train`), `merge.py`, `quantize.py`,
`smoke_generate.py`, each with a timestamped log in `logs/`, and writes `runs/smoke.json`.
About 20–30 minutes on a T4.

In [ ]:
code = sh(f"{sys.executable} harness/smoke.py --base-weights {BASE_WEIGHTS}", cwd=WORK)
assert code == 0, f"smoke run failed (exit {code}); runs/smoke.json and logs/ say where"

## Results

In [ ]:
import json
print(Path(WORK / "runs/smoke.json").read_text())

In [ ]:
rows = [json.loads(l) for l in open(WORK / "logs/smoke_generations.jsonl", encoding="utf-8")]
for i in sorted({r["row"] for r in rows}):
    print("=" * 100)
    print(rows[[r["row"] for r in rows].index(i)]["prompt"][-400:])
    for r in rows:
        if r["row"] == i:
            print(f"--- {r['model']} ({r['new_tokens']} tokens, eos={r['ended_with_eos']})")
            print(r["generation"][:600])

## Save the outputs

Everything is under `WORK`: `logs/` (raw logs, nvidia-smi samples, generations), `runs/`
(JSON), `artifacts/` (adapter, merged, merged_4bit). On Kaggle, Save Version keeps
`/kaggle/working`; download `logs/` and `runs/` at least.

In [ ]:
sh(f"tar czf {ROOT}/logs_runs_{WORK.name}.tar.gz logs runs", cwd=WORK)
print(ROOT / f"logs_runs_{WORK.name}.tar.gz")

## Budget

In [ ]:
# TODO: memory/budget.py (task 2)

## Verify

In [ ]:
# TODO: verify/ (task 3)

## Checks

In [ ]:
# TODO: checks/ (task 4); assert every check passed